## Lab 1 - Sampling

In [ ]:
from tqdm.auto import tqdm
import numpy as np
import torch
import matplotlib.pyplot as plt
from diffusers import UNet2DModel

from IPython.display import HTML
from diffusion_utilities import plot_sample
from model import ContextUnet
from train import make_noise_schedule, get_dataloader, load_checkpoint

In [5]:
print(f"Cuda available: {torch.cuda.is_available()}")

Cuda available: True


### hyperparams

In [6]:
timesteps = 500
beta1 = 1e-4
beta2 = 0.02

# network hyperparameters:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
n_features = 64
img_dim = 16  # 16x16 image size
n_channels = 3
n_context = 5

save_dir = "./checkpoints"

Using device: cuda


In [ ]:
dataloader = get_dataloader("/data/sprites/sprites_1788_16x16.npy", "/data/sprites/sprite_labels_nc_1788_16x16.npy", batch_size=64, num_workers=0)
x, labels = next(iter(dataloader))

print(f"Data points: {len(dataloader.dataset)}")

fig, axs = plt.subplots(8, 8, figsize=(8, 8))
for i, ax in enumerate(axs.flat):
    ax.imshow((x[i].permute(1, 2, 0) * 0.5 + 0.5).clamp(0, 1))
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# construct ddpm noise schedule
a_t, b_t, ab_t = make_noise_schedule(timesteps, beta1, beta2, device)

In [ ]:
def denoise_add_noise(x, t, pred_noise, z=None):
    if z is None:
        z = torch.randn_like(x)
    noise = b_t.sqrt()[t] * z
    mean = (x - pred_noise * ((1 - a_t[t]) / (1 - ab_t[t]).sqrt())) / a_t[t].sqrt()
    return mean + noise 

In [8]:
@torch.no_grad()
def sample_ddpm(model, n_sample, save_rate=20):
    """Unconditional DDPM sampling."""
    model.eval()
    samples = torch.randn(n_sample, n_channels, img_dim, img_dim, device=device)

    intermediate = []
    for t in tqdm(range(timesteps, 0, -1)):
        time = torch.tensor([t / timesteps]).to(device)
        z = torch.randn_like(samples) if t > 1 else None

        pred = model(samples, time)
        eps = pred.sample if hasattr(pred, "sample") else pred  # UNet2DModel vs ContextUnet

        samples = denoise_add_noise(samples, t, eps, z)

        if t % save_rate == 0 or t == timesteps or t < 8:
            intermediate.append(samples.detach().cpu().numpy())

    intermediate = np.stack(intermediate)
    return samples, intermediate


@torch.no_grad()
def sample_ddpm_cfg(model, n_sample, context, w=2.0, save_rate=20):
    """DDPM sampling with Classifier-Free Guidance.

    context: (n_sample, n_context) float tensor of class labels
    w: guidance scale. w=0 -> unconditional, higher -> stronger class conditioning.
    """
    model.eval()
    samples = torch.randn(n_sample, n_channels, img_dim, img_dim, device=device)
    context = context.to(device)
    uncond_context = torch.zeros_like(context)

    intermediate = []
    for t in tqdm(range(timesteps, 0, -1)):
        time = torch.tensor([t / timesteps]).to(device)
        z = torch.randn_like(samples) if t > 1 else None

        # Alternatively, could batch, but for the notebook, it's clearer
        # Batched: 1 call with 2x batch size
        # x_double   = torch.cat([x, x], dim=0)
        # ctx_double = torch.cat([context, zeros], dim=0)
        # eps_both   = model(x_double, t, context=ctx_double)
        # eps_cond, eps_uncond = eps_both.chunk(2, dim=0)
        eps_cond = model(samples, time, context=context)
        eps_uncond = model(samples, time, context=uncond_context)
        eps = eps_uncond + w * (eps_cond - eps_uncond)

        samples = denoise_add_noise(samples, t, eps, z)

        if t % save_rate == 0 or t == timesteps or t < 8:
            intermediate.append(samples.detach().cpu().numpy())

    intermediate = np.stack(intermediate)
    return samples, intermediate


In [ ]:
!ls -lh checkpoints/context_unet_c/

In [ ]:


model = ContextUnet(
    in_channels=n_channels, n_features=n_features, n_context_features=n_context, img_dim=img_dim
).to(device)

checkpoint_path = f"{save_dir}/context_unet_c_ctx/epoch_040.pth"
start_epoch, global_step, losses, optim_state_dict = load_checkpoint(checkpoint_path, model, device)


RuntimeError: Error(s) in loading state_dict for ContextUnet:
	Missing key(s) in state_dict: "init_conv.shortcut.0.weight", "init_conv.shortcut.1.weight", "init_conv.shortcut.1.bias", "init_conv.shortcut.1.running_mean", "init_conv.shortcut.1.running_var", "init_conv.conv1.0.weight", "init_conv.conv1.1.weight", "init_conv.conv1.1.bias", "init_conv.conv1.1.running_mean", "init_conv.conv1.1.running_var", "init_conv.conv2.0.weight", "init_conv.conv2.1.weight", "init_conv.conv2.1.bias", "init_conv.conv2.1.running_mean", "init_conv.conv2.1.running_var", "down1.model.0.conv1.0.weight", "down1.model.0.conv1.1.weight", "down1.model.0.conv1.1.bias", "down1.model.0.conv1.1.running_mean", "down1.model.0.conv1.1.running_var", "down1.model.0.conv2.0.weight", "down1.model.0.conv2.1.weight", "down1.model.0.conv2.1.bias", "down1.model.0.conv2.1.running_mean", "down1.model.0.conv2.1.running_var", "down1.model.1.conv1.0.weight", "down1.model.1.conv1.1.weight", "down1.model.1.conv1.1.bias", "down1.model.

In [10]:
# CFG sampling: 4 samples per class, all 5 classes
n_per_class = 4
classes = torch.eye(n_context, dtype=torch.float32)           # (5, 5) one-hot
context = classes.repeat_interleave(n_per_class, dim=0)       # (20, 5)

samples_cfg, _ = sample_ddpm_cfg(model, n_sample=n_per_class * n_context, context=context, w=2.0)

fig, axs = plt.subplots(n_context, n_per_class, figsize=(n_per_class * 2, n_context * 2))
for i, ax in enumerate(axs.flat):
    ax.imshow((samples_cfg[i].permute(1, 2, 0).cpu() * 0.5 + 0.5).clamp(0, 1))
    ax.axis("off")
for row in range(n_context):
    axs[row, 0].set_ylabel(f"class {row}", fontsize=10)
plt.suptitle("CFG samples (w=2.0)", y=1.01)
plt.tight_layout()
plt.show()


NameError: name 'model' is not defined

In [ ]:
plt.clf()
samples, intermediate_ddpm = sample_ddpm(model, 32)
animation_ddpm = plot_sample(intermediate_ddpm, 32, 4, save_dir, "ani_run", None, save=False)
HTML(animation_ddpm.to_jshtml())
